#Load

In [1]:
import os
os.getcwd()

'/content'

In [2]:
rar_path = "/content/Online Handwritten Assamese Characters Dataset.rar"

print("File exists:", os.path.exists(rar_path))
print("File size (MB):", os.path.getsize(rar_path)/1024/1024)

File exists: True
File size (MB): 7.693717956542969


In [3]:
!apt-get install unrar
!unrar x "/content/Online Handwritten Assamese Characters Dataset.rar"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 104 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/Online Handwritten Assamese Characters Dataset.rar


Would you like to replace the existing file Online Handwritten Assamese Characters Dataset/Data_Table.pdf
231604 bytes, modified on 2011-03-31 13:29
with a new one
231604 bytes, modified on 2011-03-31 13:29

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit E

All OK


In [4]:
!apt-get install -y p7zip-full

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 104 not upgraded.


In [5]:
!apt-get update -y
!apt-get install -y unrar

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (4,412 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubun

In [6]:
!mkdir -p assamese_data
!unrar x -o+ "/content/Online Handwritten Assamese Characters Dataset.rar" "assamese_data/"

Streaming output truncated to the last 5000 lines.
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/48.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/49.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/5.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/50.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/51.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/52.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/53.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Dataset/W25/54.25.txt      41%  OK 
Extracting  assamese_data/Online Handwritten Assamese Characters Datas

In [7]:
sorted(os.listdir("assamese_data/Online Handwritten Assamese Characters Dataset"))[:10]

['Data_Table.pdf',
 'W1',
 'W10',
 'W11',
 'W12',
 'W13',
 'W14',
 'W15',
 'W16',
 'W17']

## Reconstruct images

In [8]:
import os, re, math, unicodedata
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw

ROOT = "assamese_data/Online Handwritten Assamese Characters Dataset"
OUT_DIR = "cnn_data/images_128"
META_PATH = "cnn_data/metadata.csv"

IMG_SIZE = 128
PAD_FRAC = 0.08
LINE_WIDTH = 2
UPSCALE = 4

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(os.path.dirname(META_PATH)).mkdir(parents=True, exist_ok=True)

def _safe_filename(text: str) -> str:
    if text is None:
        return "NA"
    s = str(text)
    s = unicodedata.normalize("NFKC", s)

    s = re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
    return s if s else "UNK"


def _is_number_token(tok: str) -> bool:
    try:
        float(tok)
        return True
    except Exception:
        return False


def parse_handwriting_txt(path: str):

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f.readlines()]

    lines = [ln for ln in lines if ln != ""]
    if not lines:
        raise ValueError("Empty file")

    label = None

    first = lines[0]

    first_toks = re.split(r"[,\s]+", first)
    if any(re.search(r"[A-Za-z\u0980-\u09FF]", t) for t in first_toks) and not all(_is_number_token(t) for t in first_toks):
        label = first
        data_lines = lines[1:]
    else:
        data_lines = lines

    rows = []
    for ln in data_lines:
        toks = [t for t in re.split(r"[,\s]+", ln) if t != ""]
        if len(toks) < 2:
            continue
        if not (_is_number_token(toks[0]) and _is_number_token(toks[1])):
            continue
        nums = []
        for t in toks:
            if _is_number_token(t):
                nums.append(float(t))
            else:
                break

        rows.append(nums[:3])

    if not rows:
        raise ValueError("No numeric point rows found")

    max_w = max(len(r) for r in rows)
    strokes = []
    cur = []

    def flush():
        nonlocal cur
        if len(cur) >= 2:
            strokes.append(cur)
        elif len(cur) == 1:
            strokes.append(cur)
        cur = []

    if max_w >= 3:

        for r in rows:
            if len(r) < 2:
                continue
            x, y = r[0], r[1]
            p = r[2] if len(r) >= 3 else 1.0

            if x == -1 and y == -1:
                flush()
                continue
            cur.append((x, y))
            if p == 0 or p < 0:
                flush()
        flush()
    else:
        for r in rows:
            x, y = r[0], r[1]
            if x == -1 and y == -1:
                flush()
                continue
            cur.append((x, y))
        flush()

    if not strokes:
        strokes = [[(r[0], r[1]) for r in rows if len(r) >= 2 and not (r[0] == -1 and r[1] == -1)]]

    return label, strokes


def strokes_to_image(strokes, size=128, pad_frac=0.08, line_width=2, upscale=4):

    pts = [(x, y) for stroke in strokes for (x, y) in stroke if x is not None and y is not None]
    if not pts:
        return Image.new("L", (size, size), 0)

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)

    dx = maxx - minx
    dy = maxy - miny
    if dx == 0: dx = 1.0
    if dy == 0: dy = 1.0

    pad = size * pad_frac
    target = size - 2 * pad
    scale = min(target / dx, target / dy)

    cx = (minx + maxx) / 2.0
    cy = (miny + maxy) / 2.0

    def transform(p):
        x, y = p
        x = (x - cx) * scale + size / 2.0
        y = (y - cy) * scale + size / 2.0
        return (x, y)

    S = size * upscale
    img = Image.new("L", (S, S), 0)
    draw = ImageDraw.Draw(img)

    lw = max(1, int(line_width * upscale))

    for stroke in strokes:
        if not stroke:
            continue
        tpts = [transform(p) for p in stroke]
        tpts = [(x * upscale, y * upscale) for (x, y) in tpts]
        if len(tpts) == 1:
            x, y = tpts[0]
            r = lw // 2
            draw.ellipse([x - r, y - r, x + r, y + r], fill=255)
        else:
            draw.line(tpts, fill=255, width=lw, joint="curve")

    img = img.resize((size, size), resample=Image.Resampling.LANCZOS)
    return img

records = []
count_saved = 0
count_failed = 0

writers = sorted([d for d in os.listdir(ROOT) if d.startswith("W") and os.path.isdir(os.path.join(ROOT, d))])
print("Found writers:", len(writers))

for w in writers:
    w_path = os.path.join(ROOT, w)
    txt_files = sorted([f for f in os.listdir(w_path) if f.lower().endswith(".txt")])

    out_w_dir = os.path.join(OUT_DIR, w)
    os.makedirs(out_w_dir, exist_ok=True)

    saved_this_writer = 0

    for fname in txt_files:
        src_path = os.path.join(w_path, fname)
        try:
            char, strokes = parse_handwriting_txt(src_path)
            img = strokes_to_image(strokes, size=IMG_SIZE, pad_frac=PAD_FRAC, line_width=LINE_WIDTH, upscale=UPSCALE)

            char_safe = _safe_filename(char)
            out_name = f"{w}__{char_safe}__{fname.replace('.txt','')}.png"
            out_path = os.path.join(out_w_dir, out_name)

            img.save(out_path)
            count_saved += 1
            saved_this_writer += 1

            records.append({
                "writer_id": w,
                "character": char,
                "source_txt": src_path,
                "image_path": out_path
            })
        except Exception as e:
            count_failed += 1
            continue

    print(f"{w}: saved {saved_this_writer} / {len(txt_files)} images")

df = pd.DataFrame(records)
df.to_csv(META_PATH, index=False)

print("\nTOTAL saved:", count_saved)
print("TOTAL failed:", count_failed)
print("Metadata:", META_PATH)


Found writers: 45
W1: saved 183 / 183 images
W10: saved 183 / 183 images
W11: saved 183 / 183 images
W12: saved 183 / 183 images
W13: saved 183 / 183 images
W14: saved 183 / 183 images
W15: saved 183 / 183 images
W16: saved 183 / 183 images
W17: saved 183 / 183 images
W18: saved 183 / 183 images
W19: saved 183 / 183 images
W2: saved 183 / 183 images
W20: saved 183 / 183 images
W21: saved 183 / 183 images
W22: saved 183 / 183 images
W23: saved 183 / 183 images
W24: saved 183 / 183 images
W25: saved 183 / 183 images
W26: saved 183 / 183 images
W27: saved 183 / 183 images
W28: saved 183 / 183 images
W29: saved 183 / 183 images
W3: saved 183 / 183 images
W30: saved 183 / 183 images
W31: saved 183 / 183 images
W32: saved 183 / 183 images
W33: saved 183 / 183 images
W34: saved 183 / 183 images
W35: saved 183 / 183 images
W36: saved 183 / 183 images
W37: saved 183 / 183 images
W38: saved 183 / 183 images
W39: saved 183 / 183 images
W4: saved 183 / 183 images
W40: saved 183 / 183 images
W41: s

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

META_PATH = "cnn_data/metadata.csv"
df = pd.read_csv(META_PATH)

le = LabelEncoder()
df["y"] = le.fit_transform(df["writer_id"])

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["y"]
)

print("Train:", len(train_df), "Test:", len(test_df), "Classes:", df["y"].nunique())

Train: 6588 Test: 1648 Classes: 45


#CNN on Reconstructed Images (ResNet18)

In [10]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 10
LR = 3e-4
NUM_WORKERS = 2

train_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomAffine(degrees=8, translate=(0.08,0.08), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1)),
])

test_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1)),
])

class ImgDS(Dataset):
    def __init__(self, frame, tf):
        self.f = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        row = self.f.iloc[i]
        img = Image.open(row["image_path"]).convert("L")
        x = self.tf(img)
        y = int(row["y"])
        return x, y

train_loader = DataLoader(ImgDS(train_df, train_tf), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(ImgDS(test_df, test_tf), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

num_classes = df["y"].nunique()

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def eval_metrics():
    model.eval()
    correct = 0
    total = 0
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / max(1, total)

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * y.size(0)

    acc = eval_metrics()
    print(f"Epoch {epoch:02d} | loss {total_loss/len(train_df):.4f} | test acc {acc:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 01 | loss 3.4482 | test acc 0.1244
Epoch 02 | loss 2.7273 | test acc 0.2312
Epoch 03 | loss 2.2454 | test acc 0.2931


KeyboardInterrupt: 

#Feature Extraction

In [11]:
def extract_features(strokes):
    import numpy as np

    pts = np.array([p for s in strokes for p in s])

    if len(pts) < 3:
        return {k:0 for k in [
            "stroke_length","mean_speed","std_speed","max_speed",
            "mean_curvature","std_curvature","max_curvature",
            "num_strokes","pen_lifts",
            "width","height","aspect_ratio",
            "mean_direction","std_direction"
        ]}

    dx = np.diff(pts[:,0])
    dy = np.diff(pts[:,1])

    step = np.sqrt(dx**2 + dy**2)

    angles = np.arctan2(dy,dx)

    dtheta = np.diff(angles)
    dtheta = (dtheta + np.pi) % (2*np.pi) - np.pi
    curvature = np.abs(dtheta)

    width = pts[:,0].max() - pts[:,0].min()
    height = pts[:,1].max() - pts[:,1].min()

    return {
        "stroke_length": step.sum(),

        "mean_speed": step.mean(),
        "std_speed": step.std(),
        "max_speed": step.max(),

        "mean_curvature": curvature.mean() if len(curvature)>0 else 0,
        "std_curvature": curvature.std() if len(curvature)>0 else 0,
        "max_curvature": curvature.max() if len(curvature)>0 else 0,

        "mean_direction": angles.mean(),
        "std_direction": angles.std(),

        "num_strokes": len(strokes),
        "pen_lifts": len(strokes)-1,

        "width": width,
        "height": height,
        "aspect_ratio": width/(height+1e-6)
    }

#Build Feature Table

In [12]:
import os

def build_Xy(frame):
    feats = []
    ys = []
    for _, row in frame.iterrows():
        _, strokes = parse_handwriting_txt(row["source_txt"])
        feats.append(extract_features(strokes))
        ys.append(int(row["y"]))
    X = pd.DataFrame(feats).fillna(0.0)
    y = np.array(ys, dtype=int)
    return X, y

X_train, y_train = build_Xy(train_df)
X_test, y_test   = build_Xy(test_df)

print("Feature dim:", X_train.shape[1])

Feature dim: 14


#Train Models

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, n_jobs=None, multi_class="auto")),
    "SVM(RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "kNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradBoost": GradientBoostingClassifier(random_state=42),
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name:12s} | test acc = {acc:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg       | test acc = 0.2178
SVM(RBF)     | test acc = 0.2251
kNN          | test acc = 0.1456
RandomForest | test acc = 0.2021
GradBoost    | test acc = 0.1772


#Alternative Method

In [14]:
import numpy as np

def extract_features(strokes, dir_bins=16, curv_bins=12):
    pts = np.array([p for s in strokes for p in s], dtype=float)
    num_strokes = len(strokes)
    pen_lifts = max(0, num_strokes - 1)
    num_points = int(len(pts))

    if num_points < 3:
        feat = {
            "stroke_length": 0.0,
            "mean_speed": 0.0,
            "std_speed": 0.0,
            "max_speed": 0.0,
            "mean_curvature": 0.0,
            "std_curvature": 0.0,
            "max_curvature": 0.0,
            "num_strokes": float(num_strokes),
            "pen_lifts": float(pen_lifts),
            "num_points": float(num_points),
            "width": 0.0,
            "height": 0.0,
            "aspect_ratio": 1.0,
        }
        for k in range(dir_bins):
            feat[f"dir_hist_{k}"] = 0.0
        for k in range(curv_bins):
            feat[f"curv_hist_{k}"] = 0.0
        return feat

    dx = np.diff(pts[:, 0])
    dy = np.diff(pts[:, 1])
    step = np.sqrt(dx * dx + dy * dy) + 1e-12

    stroke_length = float(step.sum())
    mean_speed = float(step.mean())
    std_speed = float(step.std())
    max_speed = float(step.max())

    ang = np.arctan2(dy, dx)

    d_ang = np.diff(ang)
    d_ang = (d_ang + np.pi) % (2 * np.pi) - np.pi
    curv = np.abs(d_ang)

    mean_curv = float(curv.mean()) if curv.size else 0.0
    std_curv = float(curv.std()) if curv.size else 0.0
    max_curv = float(curv.max()) if curv.size else 0.0

    minx, maxx = float(pts[:, 0].min()), float(pts[:, 0].max())
    miny, maxy = float(pts[:, 1].min()), float(pts[:, 1].max())
    width = max(1e-6, maxx - minx)
    height = max(1e-6, maxy - miny)
    aspect_ratio = float(width / height)

    ang01 = (ang + np.pi) % (2 * np.pi)
    dir_edges = np.linspace(0.0, 2 * np.pi, dir_bins + 1)
    dir_hist, _ = np.histogram(ang01, bins=dir_edges, weights=step)
    dir_hist = dir_hist / (dir_hist.sum() + 1e-12)

    if curv.size:
        curv_clipped = np.clip(curv, 0.0, np.pi)
        w_curv = step[1:]
        curv_edges = np.linspace(0.0, np.pi, curv_bins + 1)
        curv_hist, _ = np.histogram(curv_clipped, bins=curv_edges, weights=w_curv)
        curv_hist = curv_hist / (curv_hist.sum() + 1e-12)
    else:
        curv_hist = np.zeros(curv_bins, dtype=float)

    feat = {
        "stroke_length": stroke_length,
        "mean_speed": mean_speed,
        "std_speed": std_speed,
        "max_speed": max_speed,
        "mean_curvature": mean_curv,
        "std_curvature": std_curv,
        "max_curvature": max_curv,
        "num_strokes": float(num_strokes),
        "pen_lifts": float(pen_lifts),
        "num_points": float(num_points),
        "width": float(width),
        "height": float(height),
        "aspect_ratio": aspect_ratio,
    }

    for k, v in enumerate(dir_hist):
        feat[f"dir_hist_{k}"] = float(v)

    for k, v in enumerate(curv_hist):
        feat[f"curv_hist_{k}"] = float(v)

    return feat

In [15]:
import pandas as pd
import numpy as np

def build_Xy(frame):
    feats = []
    ys = []
    for _, row in frame.iterrows():
        _, strokes = parse_handwriting_txt(row["source_txt"])
        feats.append(extract_features(strokes, dir_bins=16, curv_bins=12))
        ys.append(int(row["y"]))
    X = pd.DataFrame(feats).fillna(0.0)
    y = np.array(ys, dtype=int)
    return X, y

X_train, y_train = build_Xy(train_df)
X_test, y_test   = build_Xy(test_df)

print("Feature dim:", X_train.shape)

Feature dim: (6588, 41)


In [16]:
extract_features(strokes, dir_bins=32, curv_bins=12)

{'stroke_length': 14709.115729717509,
 'mean_speed': 43.00911032081143,
 'std_speed': 100.07513286968967,
 'max_speed': 1640.222241039306,
 'mean_curvature': 0.5237743316815174,
 'std_curvature': 0.6008233553035697,
 'max_curvature': 3.141592653589793,
 'num_strokes': 1.0,
 'pen_lifts': 0.0,
 'num_points': 343.0,
 'width': 2592.0,
 'height': 1826.0,
 'aspect_ratio': 1.4194961664841184,
 'dir_hist_0': 0.08090221206131563,
 'dir_hist_1': 0.0,
 'dir_hist_2': 0.03225939081561134,
 'dir_hist_3': 0.005096612239205423,
 'dir_hist_4': 0.020286675769221173,
 'dir_hist_5': 0.0,
 'dir_hist_6': 0.0,
 'dir_hist_7': 0.0630571895391983,
 'dir_hist_8': 0.037867674048867865,
 'dir_hist_9': 0.0,
 'dir_hist_10': 0.0,
 'dir_hist_11': 0.010193224478410786,
 'dir_hist_12': 0.04062369160784379,
 'dir_hist_13': 0.016114842044367684,
 'dir_hist_14': 0.0,
 'dir_hist_15': 0.0,
 'dir_hist_16': 0.26609349412435745,
 'dir_hist_17': 0.0,
 'dir_hist_18': 0.016144896894675603,
 'dir_hist_19': 0.02038644895682132,
 'di

In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))
svm.fit(X_train, y_train)
print("SVM tuned acc:", (svm.predict(X_test) == y_test).mean())

SVM tuned acc: 0.279126213592233
